In [ ]:
!pip install -q "pyarrow<24,>=19.0.0" transformers datasets accelerate scikit-learn
import os, re, time
from collections import Counter
import numpy as np, pandas as pd, torch
from google.colab import drive
from datasets import Dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score
from transformers import (
    AutoModelForSequenceClassification, AutoTokenizer,
    DataCollatorWithPadding, Trainer, TrainingArguments
)

drive.mount('/content/drive')
CSV_PATH = "/content/drive/MyDrive/valid.csv"
MODEL_NAME = "emilyalsentzer/Bio_ClinicalBERT"
SEED, MAX_LEN = 42, 256
torch.manual_seed(SEED); np.random.seed(SEED)

Mounted at /content/drive


In [ ]:
# 1. Exact Same Table 1 Filtered Multi-Sense Ambiguous Split (SEED = 42)
df = pd.read_csv(CSV_PATH, nrows=300000).dropna(subset=["TEXT", "LOCATION", "LABEL"])
df["LOCATION"] = df["LOCATION"].astype(int)
df["ACRONYM"] = [w[l] if 0 <= l < len(w) else "" for w, l in zip(df["TEXT"].str.split(), df["LOCATION"])]
df = df[df["ACRONYM"] != ""]

ambig = df[df["ACRONYM"].isin(df.groupby("ACRONYM")["LABEL"].nunique()[lambda x: x >= 2].index)]
ambig = ambig[ambig["LABEL"].isin(ambig["LABEL"].value_counts()[lambda x: x >= 30].index)]
top40 = ambig.groupby("ACRONYM").filter(lambda g: g["LABEL"].nunique() >= 2)["ACRONYM"].value_counts().head(40).index.tolist()

np.random.shuffle(top40)
seen_pool = ambig[ambig["ACRONYM"].isin(top40[:30])]
unseen_pool = ambig[ambig["ACRONYM"].isin(top40[30:])]

sampled_seen, _ = train_test_split(seen_pool, train_size=1500, stratify=seen_pool["LABEL"], random_state=SEED)
train_df, test_seen_df = train_test_split(sampled_seen, train_size=1000, test_size=500, stratify=sampled_seen["LABEL"], random_state=SEED)
test_unseen_df = unseen_pool.sample(n=min(500, len(unseen_pool)), random_state=SEED)

unique_labels = sorted(train_df["LABEL"].unique())
label2id = {lbl: i for i, lbl in enumerate(unique_labels)}
id2label = {i: lbl for lbl, i in label2id.items()}


In [ ]:
# 2. Format Context Window (+/- 60 Words Around LOCATION) & Tokenize
def mark_ctx(row, win=60):
    w, loc = str(row["TEXT"]).split(), int(row["LOCATION"])
    if 0 <= loc < len(w):
        w[loc] = f"[E] {w[loc]} [/E]"
        return " ".join(w[max(0, loc - win) : min(len(w), loc + win + 1)])
    return str(row["TEXT"])

tok = AutoTokenizer.from_pretrained(MODEL_NAME)
tok.add_special_tokens({"additional_special_tokens": ["[E]", "[/E]"]})

def build_ds(df_split):
    texts = [mark_ctx(r) for _, r in df_split.iterrows()]
    labels = [label2id.get(lbl, -100) for lbl in df_split["LABEL"]]
    enc = tok(texts, truncation=True, max_length=MAX_LEN)
    enc["labels"] = labels
    return Dataset.from_dict(enc)

train_ds = build_ds(train_df)
test_seen_ds = build_ds(test_seen_df)
test_unseen_ds = build_ds(test_unseen_df)


config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/213k [00:00<?, ?B/s]

In [ ]:
# 3. Train BioClinicalBERT (12 Epochs, FP16, Zero Drive Writes)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=len(unique_labels), id2label=id2label, label2id=label2id)
model.resize_token_embeddings(len(tok))

args = TrainingArguments(
    output_dir="/content/tmp_bioclinicalbert", eval_strategy="epoch", save_strategy="no",
    learning_rate=5e-5, per_device_train_batch_size=16, per_device_eval_batch_size=32,
    num_train_epochs=12, weight_decay=0.01, fp16=torch.cuda.is_available(),
    logging_steps=50, report_to="none", seed=SEED
)
trainer = Trainer(model=model, args=args, train_dataset=train_ds, eval_dataset=test_seen_ds, processing_class=tok, data_collator=DataCollatorWithPadding(tok))

torch.cuda.reset_peak_memory_stats()
t0 = time.time(); trainer.train(); train_time = time.time() - t0


pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  436MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  436MB            

model.safetensors: downloading bytes:           |  0.00B            

[transformers] BertForSequenceClassification LOAD REPORT from: emilyalsentzer/Bio_ClinicalBERT
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the chec

Epoch,Training Loss,Validation Loss
1,5.420146,5.352890
2,5.371235,5.346164
3,5.341992,5.236281
4,5.061362,5.051688
5,4.755662,4.845797
6,4.533914,4.655102
7,4.241189,4.482004
8,3.776296,4.315465
9,3.585959,4.210953
10,3.421885,4.116254


In [ ]:
# 4. Evaluate Exact Match, Macro F1 & Token F1 on Seen & Unseen Splits
norm = lambda s: re.sub(r"\s+", " ", re.sub(r"[^\w\s-]", "", str(s).lower())).strip()

def tok_f1(p, t):
    pt, tt = p.split(), t.split()
    if not pt or not tt: return float(pt == tt)
    c = sum((Counter(pt) & Counter(tt)).values())
    return (2 * c / (len(pt) + len(tt))) if c else 0.0

def evaluate_bert(hf_ds, raw_df):
    t_start = time.time()
    out = trainer.predict(hf_ds.remove_columns(["labels"]))
    elapsed = time.time() - t_start
    preds = [norm(id2label[i]) for i in np.argmax(out.predictions, axis=-1)]
    targs = [norm(l) for l in raw_df["LABEL"]]
    return (
        accuracy_score(targs, preds),
        f1_score(targs, preds, average="macro", zero_division=0),
        np.mean([tok_f1(p, t) for p, t in zip(preds, targs)]),
        elapsed
    )

s_em, s_mf1, s_tf1, s_t = evaluate_bert(test_seen_ds, test_seen_df)
u_em, u_mf1, u_tf1, u_t = evaluate_bert(test_unseen_ds, test_unseen_df)
vram = torch.cuda.max_memory_allocated() / 1024**2

print("\n" + "="*58)
print("BioClinicalBERT Baseline — Table 1 Filtered Split")
print("="*58)
print(f"Train: {train_time:.1f}s | Latency: {s_t/len(test_seen_df)*1000:.2f}ms | VRAM: {vram:.0f}MB")
print(f"Seen   -> EM: {s_em*100:.2f}% | Macro F1: {s_mf1*100:.2f}% | Token F1: {s_tf1*100:.2f}%")
print(f"Unseen -> EM: {u_em*100:.2f}% | Macro F1: {u_mf1*100:.2f}% | Token F1: {u_tf1*100:.2f}%")
print("="*58)


BioClinicalBERT Baseline — Table 1 Filtered Split
Train: 155.6s | Latency: 3.49ms | VRAM: 3100MB
Seen   -> EM: 37.00% | Macro F1: 31.96% | Token F1: 37.30%
Unseen -> EM: 0.00% | Macro F1: 0.00% | Token F1: 0.53%
